> 在线讲义：[第 29 章 · 测度论概率入门：σ-代数、Lebesgue 积分、Radon-Nikodym 导数、条件期望与收敛方式](https://wikieden.github.io/math-and-eval-stats/lessons/29-measure-probability.html) · 本书《数学与实验统计》

# 第 29 章 · 测度论概率入门：σ-代数、Lebesgue 积分、Radon-Nikodym 导数、条件期望与收敛方式

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/29-measure-probability.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 σ-代数 | 所有块的并 | `sigma_algebra(blocks)` | 8 个事件 |
| 第二步 分层蛋糕 | ∫ P(X > t) | `die_layer_cake()`、`layer_cake(expon())` | 3.5、1.0 |
| 第三步 RN 导数 | e^{x − 1/2} | `rn_derivative()` | 1.0011、1.0036 |
| 第四步 条件期望 | 块内平均 | `cond_exp_die()`、`orthogonality()` | [1.5, …]、[0, 0, 0] |
| 第五步 收敛 | 反例、LLN / CLT | `spike_sequence(n)`、`lln_clt(n)` | E = 1；0.0669 |

例子都很小：一颗骰子、一个指数分布、两个高斯、一个“几乎处处收敛但期望不收敛”的序列。
测度论的定理本身不是算出来的；这里只把每个定义落到能算的数上。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
from itertools import combinations
import numpy as np
from scipy import integrate, stats
DIE = np.arange(1, 7)

**函数 `sigma_algebra`**：由一个划分生成的 σ-代数：所有块的并（含空集和全集），共 2^块数 个事件。

In [ ]:
def sigma_algebra(blocks: list[set]) -> list[frozenset]:
    """由一个划分生成的 σ-代数：所有块的并（含空集和全集），共 2^块数 个事件。"""
    out = []
    for k in range(len(blocks) + 1):
        for c in combinations(blocks, k):
            out.append(frozenset().union(*c))
    return out

**函数 `layer_cake`**：非负随机变量 E[X] = ∫_0^∞ P(X > t) dt（“分层蛋糕”，Lebesgue 积分按值域切片的直觉）。

In [ ]:
def layer_cake(dist, upper: float = np.inf) -> float:
    """非负随机变量 E[X] = ∫_0^∞ P(X > t) dt（“分层蛋糕”，Lebesgue 积分按值域切片的直觉）。"""
    return integrate.quad(dist.sf, 0, upper)[0]

**函数 `die_layer_cake`**：骰子：E[X] = Σ_{t=0}^{5} P(X > t)，每层宽 1。

In [ ]:
def die_layer_cake() -> float:
    """骰子：E[X] = Σ_{t=0}^{5} P(X > t)，每层宽 1。"""
    return float(sum((DIE > t).mean() for t in range(6)))

**函数 `rn_derivative`**：P = N(1, 1)、Q = N(0, 1)：dP/dQ (x) = exp(x − 1/2)。用 Q 的样本检查 E_Q[dP/dQ] = 1、E_Q[X·dP/dQ] = E_P[X] = 1。

In [ ]:
def rn_derivative(n: int = 200000, seed: int = 0) -> tuple[float, float]:
    """P = N(1, 1)、Q = N(0, 1)：dP/dQ (x) = exp(x − 1/2)。用 Q 的样本检查 E_Q[dP/dQ] = 1、E_Q[X·dP/dQ] = E_P[X] = 1。"""
    x = np.random.default_rng(seed).standard_normal(n)
    w = np.exp(x - 0.5)
    return float(w.mean()), float((x * w).mean())

**函数 `cond_exp_die`**：E[X | G]：G 由划分生成时，就是在每块上取平均（X 在 G 可测函数里的最佳均方逼近）。

In [ ]:
def cond_exp_die(blocks=({1, 2}, {3, 4}, {5, 6})) -> dict[int, float]:
    """E[X | G]：G 由划分生成时，就是在每块上取平均（X 在 G 可测函数里的最佳均方逼近）。"""
    out = {}
    for b in blocks:
        m = float(np.mean(sorted(b)))
        for w in b:
            out[w] = m
    return out

**函数 `orthogonality`**：残差 X − E[X|G] 和 G 里每个事件的指示函数正交：E[(X − E[X|G])·1_B] = 0。

In [ ]:
def orthogonality(blocks=({1, 2}, {3, 4}, {5, 6})) -> list[float]:
    """残差 X − E[X|G] 和 G 里每个事件的指示函数正交：E[(X − E[X|G])·1_B] = 0。"""
    ce = cond_exp_die(blocks)
    res = np.array([w - ce[w] for w in DIE])
    return [float(np.mean(res * np.isin(DIE, sorted(b)))) for b in blocks]

**函数 `spike_sequence`**：X_n = n·1{U < 1/n}（U ~ 均匀）：P(X_n ≠ 0) = 1/n → 0（依概率收敛到 0），但 E[X_n] = 1 一直不变。返回 (P(X_n ≠ 0), E[X_n])。

In [ ]:
def spike_sequence(n: int) -> tuple[float, float]:
    """X_n = n·1{U < 1/n}（U ~ 均匀）：P(X_n ≠ 0) = 1/n → 0（依概率收敛到 0），但 E[X_n] = 1 一直不变。返回 (P(X_n ≠ 0), E[X_n])。"""
    return 1 / n, n * (1 / n)

**函数 `lln_clt`**：骰子平均：P(|均值 − 3.5| > 0.1)（大数定律：→ 0）和 √n(均值 − 3.5)/σ 的标准差（中心极限定理：→ 1）。

In [ ]:
def lln_clt(n: int, reps: int = 20000, seed: int = 1) -> tuple[float, float]:
    """骰子平均：P(|均值 − 3.5| > 0.1)（大数定律：→ 0）和 √n(均值 − 3.5)/σ 的标准差（中心极限定理：→ 1）。"""
    rng = np.random.default_rng(seed)
    m = rng.integers(1, 7, (reps, n)).mean(axis=1)
    sigma = np.sqrt(35 / 12)
    return float((np.abs(m - 3.5) > 0.1).mean()), float((np.sqrt(n) * (m - 3.5) / sigma).std())

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 σ-代数：能问概率的事件的集合")
sa = sigma_algebra([{1, 2}, {3, 4}, {5, 6}])
print(f"  只看“掷出几点在哪一对里”：{len(sa)} 个事件（2³）；完整的骰子是 2⁶ = 64 个")
print(f"  {{1, 2}} 在里面：{frozenset({1, 2}) in sa}；{{1}} 不在：{frozenset({1}) not in sa}（这个信息层次分不出 1 和 2）")

In [ ]:
print("\n第二步 Lebesgue 积分：E[X] = ∫ P(X > t) dt")
print(f"  骰子：Σ P(X > t) = {die_layer_cake():.4f}；指数分布 Exp(1)：∫ P(X > t) dt = {layer_cake(stats.expon()):.4f}")

In [ ]:
print("\n第三步 密度 = Radon-Nikodym 导数：dP/dQ")
a, b = rn_derivative()
print(f"  P = N(1, 1)、Q = N(0, 1)：dP/dQ = e^(x − 1/2)；E_Q[dP/dQ] ≈ {a:.4f}，E_Q[X·dP/dQ] ≈ {b:.4f}（= E_P[X] = 1）")

In [ ]:
print("\n第四步 条件期望：在信息允许的范围里做最好的猜测")
ce = cond_exp_die()
print(f"  E[X | 哪一对] = {[ce[w] for w in DIE]}；E[E[X|G]] = {np.mean([ce[w] for w in DIE]):.4f}")
print(f"  残差和每个块的指示函数的内积：{orthogonality()}（全是 0：条件期望是投影）")

In [ ]:
print("\n第五步 收敛方式：依概率收敛不等于期望收敛")
for n in (10, 1000, 100000):
    p, e = spike_sequence(n)
    print(f"  n = {n}：P(X_n ≠ 0) = {p:g}，E[X_n] = {e:g}")
for n in (10, 100, 1000):
    p, s = lln_clt(n)
    print(f"  骰子 n = {n}：P(|均值 − 3.5| > 0.1) = {p:.4f}，标准化后的标准差 {s:.4f}")

## 3. 代码里才看得到的

1. **`dist.sf` 就是 P(X > t)**：scipy 叫 survival function（生存函数），分层蛋糕直接对它积分。
2. **反例不用模拟**：P(X_n ≠ 0) = 1/n、E[X_n] = n × 1/n 都能直接写出来，模拟反而会因为样本里抽不到那个罕见事件而"看起来期望是 0"。
3. **CLT 检查的是标准差**：代码只看标准化后的标准差接近 1；要检查"形状是正态"，需要比分位数或画直方图。

## 4. 自己改一改

1. 把 `spike_sequence` 改成 X_n = √n·1{U < 1/n}，期望还收敛吗？
2. 用 P = N(3, 1)、Q = N(0, 1) 重跑 `rn_derivative`，看估计的方差怎么变（第 16 章：提议分布偏了，权重方差爆炸）。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> σ-代数是问卷，也是手上的信息；
> Lebesgue 按面额数钱，期望是 P(X > t) 的积分；
> 密度是换测度的汇率，重要性权重就是它；
> 条件期望是投影，残差和信息正交；
> 依概率收敛期望未必收敛，CLT 管 1/√n。